# Clase 126 — Keras preprocessing layers

Hacemos el **preprocesamiento dentro del modelo** con capas de Keras
(`Normalization`, `StringLookup`, `CategoryEncoding`, `Discretization`,
`TextVectorization`). El preprocesamiento viaja con el `.keras` y elimina el
"train-serve skew".

Requiere: `tensorflow` / `keras` (≥ 3.0).

## 1. `Normalization` con `.adapt`

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)

X_train = tf.random.normal((500, 4), mean=5.0, stddev=2.0)
norm = layers.Normalization()
norm.adapt(X_train)                       # aprende mean/std por feature
X_norm = norm(X_train)
print("mean tras normalizar ~ 0:", np.round(tf.reduce_mean(X_norm, axis=0).numpy(), 3))
print("std  tras normalizar ~ 1:", np.round(tf.math.reduce_std(X_norm, axis=0).numpy(), 3))

## 2. `StringLookup` e `IntegerLookup`

In [ ]:
categorias = tf.constant(["rojo", "verde", "azul", "rojo", "azul"])
lookup = layers.StringLookup()
lookup.adapt(categorias)
print("vocabulario:", lookup.get_vocabulary())            # [OOV, ...]
print("rojo/verde/negro ->", lookup(["rojo", "verde", "negro"]).numpy())  # negro=OOV=0

int_lookup = layers.IntegerLookup()
int_lookup.adapt(tf.constant([10, 20, 30, 20]))
print("IntegerLookup 20/99 ->", int_lookup([20, 99]).numpy())

## 3. `CategoryEncoding` y `Discretization`

In [ ]:
one_hot = layers.CategoryEncoding(num_tokens=4, output_mode="one_hot")
print("one-hot de [0, 2, 3]:\n", one_hot([0, 2, 3]).numpy())

multi = layers.CategoryEncoding(num_tokens=4, output_mode="multi_hot")
print("multi-hot de [[0, 1], [2, 2]]:\n", multi([[0, 1], [2, 2]]).numpy())

disc = layers.Discretization(bin_boundaries=[0.0, 1.0, 2.0])
print("Discretization de [-0.5, 0.5, 1.5, 3.0]:", disc([-0.5, 0.5, 1.5, 3.0]).numpy())

## 4. `TextVectorization` (tokeniza + indexa)

In [ ]:
textos = tf.constant([
    "esta pelicula es excelente",
    "muy mala y aburrida",
    "excelente actuacion",
])
vectorizar = layers.TextVectorization(max_tokens=20, output_mode="int",
                                      output_sequence_length=5)
vectorizar.adapt(textos)
print("vocabulario (primeros 8):", vectorizar.get_vocabulary()[:8])
print("secuencias:\n", vectorizar(textos).numpy())

## 5. Modelo end-to-end (preprocesamiento DENTRO del modelo)

In [ ]:
X_train = tf.random.normal((500, 4), mean=5.0, stddev=2.0)
y_train = tf.cast(tf.reduce_sum(X_train, axis=1) > 20.0, tf.int32)

norm = layers.Normalization()
norm.adapt(X_train)                        # adaptar SOLO con train (evita leakage)

entradas = keras.Input(shape=(4,))
x = norm(entradas)                         # normalización dentro del grafo
x = layers.Dense(32, activation="relu")(x)
salidas = layers.Dense(1, activation="sigmoid")(x)
modelo = keras.Model(entradas, salidas)
modelo.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
modelo.fit(X_train, y_train, epochs=3, verbose=2)

## 6. Guardar y predecir sobre datos RAW

In [ ]:
modelo.save("modelo_e2e.keras")
recargado = keras.models.load_model("modelo_e2e.keras")

crudo = tf.constant([[5.0, 5.0, 5.0, 5.0]])   # datos RAW, sin escalar a mano
p1 = modelo.predict(crudo, verbose=0)
p2 = recargado.predict(crudo, verbose=0)
print("predicción sobre RAW (sin scaler externo):", float(p1))
print("idéntica tras recargar:", np.allclose(p1, p2))

## Ejercicios

1. **Normalization tabular**: `norm.adapt(X_train)`; verificá que `mean ~ 0` y
   `std ~ 1` en el output.
2. **StringLookup**: adaptá con un array de categorías y mapeá algunas nuevas
   (OOV) a índice 0.
3. **Modelo end-to-end**: meté la capa `Normalization` dentro del `keras.Model`
   y adaptala solo con el train set.
4. **TextVectorization**: tokenizá reseñas (IMDB) y entrená un modelo de
   sentimiento.
5. **Hashing**: con muchas categorías únicas, comparación de `Hashing(1024)` vs
   `StringLookup` con vocabulario truncado.

## Conclusiones

- `.adapt(data)` aprende los parámetros (mean/std, vocabulario) — reemplaza el fit-transform separado; adaptar **solo** con train.
- `Normalization` escala; `StringLookup`/`IntegerLookup` indexan categóricas; `CategoryEncoding` hace one/multi-hot; `Discretization` bucketiza.
- `TextVectorization` tokeniza e indexa texto en una sola capa.
- Metiendo las capas **dentro** del modelo, `model.predict(raw_data)` funciona sin scaler externo: adiós al train-serve skew.
- El `.keras` guarda arquitectura + pesos + preprocesamiento en un solo archivo.

## ✅ Soluciones de los ejercicios

Keras preprocessing layers (cap. 13). Se validan por AST sin TF; el **Ej. 1** (Normalization) genera datos con NumPy para verificar mean≈0/std≈1. Cubren `Normalization`, `StringLookup`, un modelo end-to-end con el preprocesado embebido, `TextVectorization` y `Hashing`.

**Ej. 1 — Normalization tabular.** `adapt(X_train)` aprende mean/var; la salida queda con mean≈0, std≈1.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Normalization

X_train = (np.random.randn(1000, 5) * 3 + 7).astype("float32")
norm = Normalization()
norm.adapt(X_train)                       # aprende estadisticas del TRAIN
X_norm = norm(X_train)
print("mean ~0:", float(tf.reduce_mean(X_norm)))
print("std  ~1:", float(tf.math.reduce_std(X_norm)))

**Ej. 2 — StringLookup.** `adapt(categorias)` construye el vocab; mapea strings -> ints (0 = OOV).

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import StringLookup

categorias = tf.constant(["A", "B", "C", "A", "D"])
lookup = StringLookup()
lookup.adapt(categorias)
print("A,B,C ->", lookup(["A", "B", "C"]).numpy())   # enteros
print("vocab:", lookup.get_vocabulary())              # ['[UNK]', ...]

**Ej. 3 — Modelo end-to-end.** La capa `Normalization` vive DENTRO del modelo (adapt antes de entrenar).

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras

X_train = np.random.randn(500, 8).astype("float32")
norm = keras.layers.Normalization()
norm.adapt(X_train)                        # adaptar ANTES de construir/entrenar

inputs = keras.Input((8,))
x = norm(inputs)                           # el preprocesado forma parte del grafo
x = keras.layers.Dense(64, activation="relu")(x)
out = keras.layers.Dense(1)(x)
model = keras.Model(inputs, out)
# ventaja: el modelo guardado ya normaliza en produccion (no hay pipeline externo que sincronizar)
print("output_shape:", model.output_shape)

**Ej. 4 — TextVectorization.** Tokeniza texto crudo dentro del modelo (pipeline de sentimiento IMDB).

In [ ]:
import tensorflow as tf
from tensorflow import keras

vec = keras.layers.TextVectorization(max_tokens=10_000, output_sequence_length=200)
# vec.adapt(train_texts)    # construye el vocabulario desde el corpus de IMDB

inputs = keras.Input(shape=(1,), dtype=tf.string)
x = vec(inputs)                                  # texto -> secuencia de ints
x = keras.layers.Embedding(10_000, 32)(x)
x = keras.layers.GlobalAveragePooling1D()(x)
out = keras.layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, out)
print("modelo de sentimiento con TextVectorization embebido (acepta strings crudos)")

**Ej. 5 — Hashing.** `Hashing(1024)` mapea 100k+ categorías a 1024 buckets sin vocab (a costa de colisiones).

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Hashing

hashing = Hashing(num_bins=1024)                # sin adapt, sin vocabulario
ids = hashing(tf.constant(["user_9182", "item_55", "user_9182"]))
print(ids.numpy())        # mismo string -> mismo bucket
# vs StringLookup: Hashing no necesita ver el vocab (ideal para cardinalidad enorme),
# pero dos categorias pueden colisionar en el mismo bin.